## Step 1. Preprocessing (removing transparent area)

### Load packages & modules

In [1]:

import os
import shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt

from PIL import Image

def save_png_with_black_alpha(input_path, output_path):
    """
    Reads a PNG file, converts its transparent areas (alpha channel) to black,
    and saves it as a new RGB PNG file.
    
    Args:
        input_path (str): Path to the input image file.
        output_path (str): Path where the processed image will be saved.
    """
    # 1. Load the image in RGBA mode using Pillow and convert to a NumPy array
    img_pil = Image.open(input_path).convert("RGBA")
    img_np = np.array(img_pil)  # Shape: H x W x 4

    # 2. Extract RGB channels and normalize the alpha channel
    rgb = img_np[..., :3].astype(np.uint8)
    alpha = img_np[..., 3] / 255.0  # Normalize to range 0.0 ~ 1.0

    # 3. Apply alpha blending to turn transparent areas (alpha=0) into black
    rgb = (rgb * alpha[..., None]).astype(np.uint8)

    # 4. Remove the alpha channel and convert back to RGB mode
    img_out = Image.fromarray(rgb, mode="RGB")

    # 5. Save the processed image as a new PNG file
    img_out.save(output_path, format="PNG")


def preprocessing_images(base_path, target_path):
    """
    Recursively walks through the base directory, filters all PNG images,
    processes them to replace transparency with black, and saves them
    into the target directory while preserving the original folder structure.
    
    Args:
        base_path (str): The root directory containing source PNG images.
        target_path (str): The destination root directory for processed images.
    """
    for root, dirs, files in os.walk(base_path):
        # Replicate the original subdirectory structure in the target path
        relative_path = os.path.relpath(root, base_path)
        target_dir = os.path.join(target_path, relative_path)

        for file in files:
            # Process only PNG files
            if not file.endswith(('.png')):
                continue
                
            src_file = os.path.join(root, file)
            dst_file = os.path.join(target_dir, file)
            
            # Ensure the target directory exists before saving
            os.makedirs(target_dir, exist_ok=True)

            # Process and save the image with transparency replaced by black
            save_png_with_black_alpha(src_file, dst_file)

### Conduct Preprocessing

In [2]:
base_path = '../data/upload'
target_path = '../data/preprocessed'

preprocessing_images(base_path, target_path)

## Step 2. Detecting Adult or Juveniles from the given preprocessed images

### Configuration and Initialization (Common)

In [3]:
import os
import torch
print(torch.cuda.is_available())

from ultralytics import YOLO
import cv2

True


### Configuration and Initialization (Adult Detection)

In [6]:
target = 'adult'
model = YOLO(f'../model/detector_{target}.pt')  # Load the custom YOLO model
conf = 0.05
imgsz = 1280
iou = 0.30

# Root folder containing the target images to run predictions on
source_path = os.path.join('../data/preprocessed', target)
detection_path = '../data/detection'  # Directory where final results will be saved


### Configuration and Initialization (Juvenile Detection)

In [8]:
target = 'juvenile'
model = YOLO(f'../model/detector_{target}.pt')  # Load the custom YOLO model
conf = 0.01
imgsz = 1280
iou = 0.10

# Root folder containing the target images to run predictions on
source_path = os.path.join('../data/preprocessed', target)
detection_path = '../data/detection'  # Directory where final results will be saved


### Detection Loop

In [11]:
for root, dirs, files in os.walk(source_path):
    # Count the number of image files in the current directory
    img_cnt = len(list(filter(lambda f: f.endswith('png') or f.endswith('jpg'), files)))
    
    # Run prediction only if images exist in the folder
    if img_cnt > 0:
        relative_path = os.path.relpath(root, source_path)
        save_path = os.path.join(detection_path, relative_path)
        
        # Run YOLO inference on the entire directory
        results = model.predict(
            source=root, imgsz=imgsz, iou=iou, conf=conf, 
            save=False, save_crop=True, save_txt=True, 
            show_conf=False, show_labels=False, 
            project=save_path, name=target
        )
        
        # Process detection results for each image
        for result in results:
            img_path = result.path
            label_path = os.path.join(save_path, target, 'labels')
            
            # Map the image filename to its corresponding YOLO txt label file
            txt_name = os.path.splitext(os.path.basename(img_path))[0] + ".txt"            
            txt_path = os.path.join(label_path, txt_name)
            
            # Skip visualization if no bounding box labels were generated
            if not os.path.exists(txt_path):
                continue
        
            # Load the original image to draw bounding boxes
            img = cv2.imread(img_path)
            H, W = img.shape[:2]
        
            # Read the predicted labels (YOLO format: class_id x_center y_center width height)
            with open(txt_path, "r") as f:
                lines = f.readlines()
        
            # Draw bounding boxes and crop index numbers on the image
            for idx, line in enumerate(lines):
                class_id, x_center, y_center, w, h = map(float, line.strip().split())
        
                # Convert normalized YOLO coordinates back to pixel coordinates
                x1 = int((x_center - w/2) * W)
                y1 = int((y_center - h/2) * H)
                x2 = int((x_center + w/2) * W)
                y2 = int((y_center + h/2) * H)
        
                # Draw the bounding box (Green rectangle)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
                # Label the box with its sequence number (Red text)
                cv2.putText(img, f"{idx+1}", (x1, y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        
            # Save the annotated image to the "labeled" directory
            save_img_path = os.path.join(save_path, target, "labeled", os.path.basename(img_path))
            os.makedirs(os.path.dirname(save_img_path), exist_ok=True)
            cv2.imwrite(save_img_path, img)


image 1/1 /ws/microorganism-detection/alkimi-size/script/../data/preprocessed/juvenile/Cd/sample_cd_juvenile.png: 1280x1280 68 larvas, 5.4ms
Speed: 8.5ms preprocess, 5.4ms inference, 1.2ms postprocess per image at shape (1, 3, 1280, 1280)
Results saved to ../data/detection/Cd/juvenile3
1 label saved to ../data/detection/Cd/juvenile3/labels

image 1/1 /ws/microorganism-detection/alkimi-size/script/../data/preprocessed/juvenile/Cu/sample_cu_juvenile.png: 1280x1280 61 larvas, 4.3ms
Speed: 7.6ms preprocess, 4.3ms inference, 1.0ms postprocess per image at shape (1, 3, 1280, 1280)
Results saved to ../data/detection/Cu/juvenile3
1 label saved to ../data/detection/Cu/juvenile3/labels

image 1/1 /ws/microorganism-detection/alkimi-size/script/../data/preprocessed/juvenile/As/sample_as_juvenile.png: 1280x1280 44 larvas, 4.3ms
Speed: 7.7ms preprocess, 4.3ms inference, 1.0ms postprocess per image at shape (1, 3, 1280, 1280)
Results saved to ../data/detection/As/juvenile3
1 label saved to ../data/d

## Step 3. Segmentation & Body size calculation

### Load packages & modules

In [16]:
from ultralytics import YOLO
import pyclesperanto_prototype as cle
import matplotlib.pyplot as plt
from skimage.color import rgb2gray
import os
from tqdm import tqdm
import numpy as np
from scipy.ndimage import center_of_mass
import cv2

  
def calculate_euclidean_distance (labels_cpu, image_file):
    """
    Finds the labeled object closest to the center of the image using Euclidean distance.
    
    Args:
        labels_cpu (numpy.ndarray): Labeled image array on CPU.
        image_file (str): Filename for error logging reference.
        
    Returns:
        tuple: (center_object_mask, max_count) where center_object_mask is a boolean mask 
               of the closest object, and max_count is its pixel count. Returns (None, None) if nothing found.
    """
    # Initialize variables to track the closest object
    min_distance = float("inf")
    closest_label = None
        
    # Calculate the centroid (center of mass) for each unique label
    unique_labels, counts = np.unique(labels_cpu, return_counts=True)
    centroids = {label: center_of_mass(labels_cpu == label) for label in unique_labels if label != 0}
    
    # Calculate the absolute center coordinates of the image
    h, w = labels_cpu.shape  
    center_x, center_y = w / 2, h / 2  
        
    # Find the label with the minimum Euclidean distance to the image center
    for label, (cy, cx) in centroids.items():
        distance = np.linalg.norm([cx - center_x, cy - center_y])  # Euclidean distance
        if distance < min_distance:
            min_distance = distance
            closest_label = label

    # Generate a mask for the object closest to the center
    if closest_label is not None:
        center_object_mask = labels_cpu == closest_label
        max_count = counts[closest_label]
    else:
        print(f"[WARN] No objects detected in {image_file}.")
        return None, None
        
    return center_object_mask, max_count


def plt_images(image, save_path, center_object_mask, max_count, image_file):               
    """
    Plots the original image alongside the segmented mask side-by-side and saves the figure.
    """
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    
    # Plot original grayscale image
    ax[0].imshow(image, cmap='gray')
    ax[0].set_title('Original Image')
    ax[0].axis('off')

    # Plot segmented binary mask with area stats (assuming 1 pixel = 0.012 mm^2)
    ax[1].imshow(center_object_mask, cmap='gray')
    ax[1].set_title(f'Pixels:{max_count}, Size: {max_count * 0.012:.3f} mm^2')
    ax[1].axis('off')

    # Save the figure with minimal padding
    plt.savefig(os.path.join(save_path, f'seg_{image_file}'), bbox_inches='tight', pad_inches=0)
    plt.close()


def segmentation(target_label, source_path, target_path, save_root_path, threshold, spot_sigma, outline_sigma):
    """
    Performs segmentation for 'adult' or 'juvenile' target cropped images using Voronoi-Otsu labeling.
    Filters paths to only process target adult (or juvenile) crops.
    """
    for root, dirs, files in os.walk(source_path):
        # Target specific directory paths (must contain target_path, 'crops', and target_label)
        if target_path not in root:
            continue
        if 'crops' not in root or target_label not in root:
            continue
            
        image_files = [f for f in files if f.endswith(('.jpg', '.jpeg', '.png'))]
        
        # Determine the destination path based on the input folder structure
        abs_path = root.split('/crops')[0]
        rel_path = os.path.relpath(abs_path, source_path)
        save_path = os.path.join(save_root_path, rel_path)
        os.makedirs(save_path, exist_ok=True)
            
        for image_file in tqdm(image_files):
            img_path = os.path.join(root, image_file)
            image = cv2.imread(img_path)
    
            # Convert to grayscale if it's a color image
            if image.ndim == 3:
                image = rgb2gray(image)
    
            # Push thresholded image to GPU memory
            img_bn = cle.push(image > threshold) 
            
            # Pull binary mask back to check if it's completely empty
            binary_array = cle.pull(img_bn)
            if np.all(binary_array == 0):
                print(f"[WARN] No object found in thresholded binary array -> Skipping segmentation for {image_file}")
                continue
            
            # Apply GPU-accelerated Voronoi-Otsu labeling
            labels = cle.voronoi_otsu_labeling(img_bn, spot_sigma=spot_sigma, outline_sigma=outline_sigma) 
    
            # Pull the resulting labeled array back to CPU
            labels_cpu = cle.pull(labels)
            
            # Identify the primary object closest to the center
            center_object_mask, max_count = calculate_euclidean_distance(labels_cpu, image_file) 
            
            # If valid object found, save the visual side-by-side comparison
            if max_count is not None:            
                plt_images(image, save_path, center_object_mask, max_count, image_file)

### Configuration and Initialization (Common)

In [13]:
detection_path = '../data/detection'
seg_path = '../data/segmentation'

### Run Segmentation (Adult, As)

In [17]:
target_label = 'adult'
threshold = 0.65
spot_sigma = 5.5
outline_sigma = 0.5

segmentation(target_label, detection_path, 'As', seg_path, threshold=threshold, spot_sigma=spot_sigma, outline_sigma=outline_sigma)

0it [00:00, ?it/s]
100%|██████████| 10/10 [00:01<00:00,  5.98it/s]


### Run Segmentation (Adult, Cd)

In [15]:
target_label = 'adult'
threshold = 0.6
spot_sigma = 5.5
outline_sigma = 0.5

segmentation(target_label, detection_path, 'Cd', seg_path, threshold=threshold, spot_sigma=spot_sigma, outline_sigma=outline_sigma)

0it [00:00, ?it/s]
100%|██████████| 10/10 [00:03<00:00,  2.92it/s]


### Run Segmentation (Adult, Cu)

In [36]:
target_label = 'adult'
threshold = 0.64
spot_sigma = 5.5
outline_sigma = 0.5

segmentation(target_label, detection_path, 'Cu', seg_path, threshold=threshold, spot_sigma=spot_sigma, outline_sigma=outline_sigma)

0it [00:00, ?it/s]
100%|██████████| 10/10 [00:03<00:00,  3.30it/s]


### Run Segmentation (Juvenile, As)

In [37]:
target_label = 'juvenile'
threshold = 0.6
spot_sigma = 3
outline_sigma = 0.8

segmentation(target_label, detection_path, 'As', seg_path, threshold=threshold, spot_sigma=spot_sigma, outline_sigma=outline_sigma)

0it [00:00, ?it/s]
 30%|██▉       | 13/44 [00:03<00:04,  6.89it/s]

[WARN] No object found in thresholded binary array -> Skipping segmentation for sample_as_juvenile40.jpg


 55%|█████▍    | 24/44 [00:04<00:02,  8.34it/s]

[WARN] No object found in thresholded binary array -> Skipping segmentation for sample_as_juvenile42.jpg


 57%|█████▋    | 25/44 [00:04<00:02,  8.31it/s]

...

 77%|███████▋  | 34/44 [00:05<00:01,  8.79it/s]

[WARN] No object found in thresholded binary array -> Skipping segmentation for sample_as_juvenile39.jpg


 98%|█████████▊| 43/44 [00:07<00:00,  9.69it/s]

[WARN] No object found in thresholded binary array -> Skipping segmentation for sample_as_juvenile35.jpg


100%|██████████| 44/44 [00:07<00:00,  6.13it/s]


### Run Segmentation (Juvenile, Cd)

In [40]:
target_label = 'juvenile'
threshold = 0.5
spot_sigma = 3
outline_sigma = 0.5

segmentation(target_label, detection_path, 'Cd', seg_path, threshold=threshold, spot_sigma=spot_sigma, outline_sigma=outline_sigma)

0it [00:00, ?it/s]
100%|██████████| 68/68 [00:08<00:00,  7.60it/s]


### Run Segmentation (Juvenile, Cu)

In [41]:
target_label = 'juvenile'
threshold = 0.5
spot_sigma = 3
outline_sigma = 0.5

segmentation(target_label, detection_path, 'Cu', seg_path, threshold=threshold, spot_sigma=spot_sigma, outline_sigma=outline_sigma)

0it [00:00, ?it/s]
 11%|█▏        | 7/61 [00:01<00:06,  7.86it/s]

[WARN] No object found in thresholded binary array -> Skipping segmentation for sample_cu_juvenile59.jpg


100%|██████████| 61/61 [00:11<00:00,  5.24it/s]
